# Install necessary libraries

In [71]:
%pip install --upgrade --quiet dotenv pydantic typing-extensions langgraph google-generativeai rich

# Import modules

In [2]:
from google import genai
from google.colab import userdata
import os
from typing import Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
import google.generativeai as genai
from dotenv import load_dotenv
from typing import TypedDict, Optional, Literal
from typing_extensions import TypedDict
import json
from pydantic import BaseModel

# Load API key and configure genai

In [3]:
# Make sure you have a .env file with your GEMINI_API_KEY
load_dotenv(override=True)

# Get the Gemini API key from environment variables
API_KEY = userdata.get('GOOGLE_API_KEY_2')

# Configure the genai library with the API key
genai.configure(api_key=API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash')

# Configure genai and initialize models

In [4]:
genai.configure(api_key=API_KEY)
json_model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config=genai.types.GenerationConfig(
        response_mime_type="application/json"
    )
)
# A standard model for general text generation.
text_model = genai.GenerativeModel('gemini-2.5-flash')

# Define Pydantic model and State TypedDict

In [5]:
class ClassifyMessageResponse(BaseModel):
    query_type: Literal["order_status",
                        "return_policy", "product_info", "other"]

class State(TypedDict):
    user_query: str
    query_type: Optional[str]
    llm_result: Optional[str]

# Define graph node functions

In [6]:

def classify_message(state: State):
    """Detects if the user's query is about coding, cooking, or general topics."""
    print("---  classifying message ---")
    query = state["user_query"]
    prompt = f"""
    You are a helpful Customer Support Agent for a Small E-commerce Store.
    Your job is to detect if the customer's query is related to order_status,
    return_policy, product_info, or other.
    Return your response as a JSON object with a single key "query_type".
    Example: {{"query_type": "order_status"}}

    User Query: "{query}"
    """
    response = json_model.generate_content(prompt)
    parsed_response = json.loads(response.text)
    state["query_type"] = parsed_response["query_type"]
    print(f"   - Classified as: {state['query_type']}")
    return state


def order_status(state: State):
    """Handles order status queries."""
    print("--- order status agent ---")
    query = state["user_query"]
    # We create a model with a specific system instruction for this task.
    agent = genai.GenerativeModel(
        'gemini-2.5-flash',
        system_instruction="Provide short, precise information about customers's order status query."
    )
    response = agent.generate_content(query)
    state["llm_result"] = response.text
    return state


def return_policy(state: State):
    """Handles  return policy queries."""
    print("--- return policy agent ---")
    query = state["user_query"]
    # We create a model with a specific system instruction for this task.
    agent = genai.GenerativeModel(
        'gemini-2.5-flash',
        system_instruction="Answer the customers's order's return_policy query with link."
    )
    response = agent.generate_content(query)
    state["llm_result"] = response.text
    return state


def product_info(state: State):
    """Handles product info queries."""
    print("--- product info agent ---")
    query = state["user_query"]
    # We create a model with a specific system instruction for this task.
    agent = genai.GenerativeModel(
        'gemini-2.5-flash',
        system_instruction="Provide short, precise information about provided product's information (i.e Features,warranty,stock availability,etc)."
    )
    response = agent.generate_content(query)
    state["llm_result"] = response.text
    return state


def other(state: State):
    """Handles other queries."""
    print("--- other agent ---")
    query = state["user_query"]
    # We create a model with a specific system instruction for this task.
    agent = genai.GenerativeModel(
        'gemini-2.5-flash',
        system_instruction="Provide short, precise information about the customer's general query regarding online purchase."
    )
    response = agent.generate_content(query)
    state["llm_result"] = response.text
    return state

# Define routing function

In [7]:
# -> Literal["order_status", "return_policy", "product_info", "other"]:
def route_query(state: State):
    """Routes the query to the correct agent based on classification."""
    return state["query_type"]

# Build the graph

In [8]:
# --- Graph Definition (Unchanged) ---
graph_builder = StateGraph(State)

# Add all nodes
# ["order_status", "return_policy", "product_info", "other"]
graph_builder.add_node("classify_message", classify_message)
graph_builder.add_node("order_status", order_status)
graph_builder.add_node("return_policy", return_policy)
graph_builder.add_node("product_info", product_info)
graph_builder.add_node("other", other)

# Wire the graph

In [9]:
# Wire the graph
graph_builder.add_edge(START, "classify_message")
graph_builder.add_conditional_edges("classify_message",
                                    route_query,
                                    {"order_status": "order_status",
                                     "return_policy": "return_policy",
                                     "product_info": "product_info",
                                     "other": "other"}
                                    )
graph_builder.add_edge("other", END)
graph_builder.add_edge("order_status", END)
graph_builder.add_edge("return_policy", END)
graph_builder.add_edge("product_info", END)


graph = graph_builder.compile()

# Define main function

In [15]:
from rich.text import Text
from rich.console import Console
from rich.panel import Panel
from rich import print

console = Console()

def main():
    console.print(Panel("🤖 Welcome to the Customer Support Chatbot!", title="[bold blue]Customer Support[/bold blue]", expand=False))
    while True:  # Add a loop for continuous interaction
        user = console.input("[bold green]Ask your query (type 'exit', 'stop', or 'quit' to quit):[/bold green] ")
        if user.lower() in ["exit", "stop", "quit"]:
            console.print("[bold green]Exiting chatbot. Goodbye![/bold green]")
            break  # Exit the loop if user types 'exit' or 'stop'

        # Define the initial state
        initial_state: State = {
            "user_query": user,
            "llm_result": None,
            "query_type": None,
        }
        console.print(f"\n[bold blue]Processing your query...[/bold blue]")
        # Invoke the graph
        graph_result = graph.invoke(initial_state)

        query_type_text = Text(f"Query Type: ", style="bold")
        query_type_text.append(Text(f"{graph_result.get('query_type', 'N/A')}", style="bold green"))

        result_text = Text(f"Result: ", style="bold")
        result_text.append(Text(f"{graph_result.get('llm_result', 'No result')}"))

        # Combine texts with a newline character
        combined_text = Text.assemble(query_type_text, "\n", result_text)

        console.print(Panel(combined_text, title="[bold blue]🧠 Final Response from Graph[/bold blue]", expand=False))

In [11]:
# def main():
#     print("Ask your query")
#     user = input("> ")
#     # Define the initial state
#     initial_state: State = {
#         "user_query": user,
#         "llm_result": None,
#         "query_type": None,
#     }
#     # Invoke the graph
#     graph_result = graph.invoke(initial_state)

#     print("\n" + "="*30)
#     print("🧠 Final Response from Graph:")
#     print(f"Query Type: {graph_result.get('query_type', 'N/A')}")
#     print(f"Result: {graph_result.get('llm_result', 'No result')}")
#     print("="*30)


# Run the chatbot

In [16]:
if __name__ == "__main__":
    main()

╭───────────── Customer Support ──────────────╮
│ 🤖 Welcome to the Customer Support Chatbot! │
╰─────────────────────────────────────────────╯

Ask your query (type 'exit', 'stop', or 'quit' to quit):

price of iphone 16


Processing your query...

---  classifying message ---

...

--- other agent ---

╭─ 🧠 Final Response from Graph ─╮
│ Query Type: other              │
│ Result: Order Status           │
╰────────────────────────────────╯

Ask your query (type 'exit', 'stop', or 'quit' to quit):

stop


Exiting chatbot. Goodbye!